In [1]:
import sys, os, json, gc
from pathlib import Path
from collections import Counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("python:", sys.executable)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
DATA = PROJECT / "data"
CHECKPOINTS = PROJECT / "checkpoints"

ADAPTER_3B_V3 = CHECKPOINTS / "qwen3b-qlora-v3_bad_do_not_use" / "checkpoint-final"

assert (DATA / "train_v2.jsonl").exists()
assert (DATA / "test.jsonl").exists()
print("paths ok")

python: C:\Users\shlok\projects\ddp-llm\.venv\Scripts\python.exe
torch: 2.11.0+cu128
cuda: True
paths ok


In [2]:
# v4 = v3 + explicit clause about bare confirmation words.
# The v2 model with prompt v3 handles quality/reaction words ("great", "perfect")
# but fails on confirmation/agreement words ("yes", "definitely", "bingo") because
# those pattern-match to "user is confirming a choice they must have made" and
# the model picks A or B arbitrarily. This clause makes the distinction explicit.

SYSTEM_PROMPT_V4 = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.
- CRITICAL RULE 1 - reaction words: If the utterance is only a short positive reaction that does NOT reference any option (e.g. "works great", "perfect", "nice one", "go with it", "that works", "cool", "awesome"), output "*". A positive word alone is not enough.
- CRITICAL RULE 2 - confirmation words: Bare confirmation or agreement words alone (e.g. "yes", "yeah", "yep", "definitely", "for sure", "absolutely yes", "exactly", "bingo", "spot on", "right on", "100%") are NOT endorsements when used alone. They only count as an endorsement if they name an option (e.g. "yes A", "definitely B", "bingo, the left one"). If no option is named, output "*".
- To count as an endorsement, the user MUST reference the options - either by letter (A, B, C, D), position (left, right, first, second, middle, last), quantifier over options (all, both, either, these, the ones, all four, both of them), or description that clearly points at an option.

Output ONLY the JSON. No explanation, no prose."""

print(f"v4 prompt: {len(SYSTEM_PROMPT_V4)} chars")

v4 prompt: 1908 chars


In [3]:
def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]


def build_model(base_model_id, adapter_path):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    tok = AutoTokenizer.from_pretrained(base_model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        base_model_id, quantization_config=bnb, device_map="cuda",
    )
    m = PeftModel.from_pretrained(base, str(adapter_path))
    m.eval()
    return m, tok


def parse(model, tok, utterance, options, system_prompt=SYSTEM_PROMPT_V4):
    options_str = "[" + ", ".join(f'"{o}"' for o in options) + "]"
    user_msg = f"Options: {options_str}\nUser: {utterance}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_msg},
    ]
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=15, do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    raw = tok.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
    ).strip()
    if raw.startswith("```"):
        raw = raw.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        return None, raw
    return parsed, raw


def labels_equal(a, b):
    if a == "*" or b == "*":
        return a == b
    if isinstance(a, list) and isinstance(b, list):
        return set(a) == set(b)
    return False


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB", flush=True)


print("helpers defined")

helpers defined


In [4]:
# ~150 new examples targeting the confirmation-word failure mode discovered
# after v2. Three sub-groups:
# 1) confirmation_bare: bare "yes"/"definitely"/etc. -> "*"  (60)
# 2) confirmation_with_referent: same words + a letter/position -> endorsement (60)
#    This prevents the model from overfitting to "confirmation word = *".
# 3) ambient_reinforce: more variations of the v2 ambient patterns (30).

NEW_EXAMPLES_V3 = []

# --- 1) confirmation_bare -> "*" ---
CONF_BARE = [
    # plain agreement
    "yes", "yeah", "yep", "yup", "yeah yeah", "yes yes", "mm yes",
    "of course", "sure", "sure thing", "for sure", "of course yes",
    # emphasis / intensifiers
    "definitely", "definitely!", "absolutely", "absolutely yes",
    "100%", "totally", "totally yes", "no doubt", "for real",
    "no question", "without a doubt", "no question about it",
    # affirmation phrases
    "exactly", "exactly right", "spot on", "right on", "bang on",
    "bingo", "nailed it", "you got it", "correct", "that's right",
    "you're right", "true that", "true", "word", "facts",
    # casual agreement
    "aye", "roger that", "affirmative", "agreed", "sold",
    "count me in", "im in", "sign me up", "sold on it",
    # ambiguous fillers that read as agreement
    "hm yeah", "mm hmm", "uh huh", "mhm", "ok yeah", "yeah ok",
    "right yeah", "yeah right", "totally agree", "same",
    "hard agree", "co-signed",
]
for utt in CONF_BARE:
    NEW_EXAMPLES_V3.append({
        "utterance": utt,
        "options": ["A", "B", "C", "D"],
        "label": "*",
        "category": "confirmation_bare",
    })

# --- 2) confirmation_with_referent -> endorsement ---
# CRITICAL: same words + letter/position must still endorse. Prevents the
# model from learning "yes/definitely = *" as a hard rule.
CONF_WITH_REF_2OPT = [
    ("yes A", ["A"]), ("yes B", ["B"]),
    ("yeah A", ["A"]), ("yeah B", ["B"]),
    ("yep A", ["A"]), ("A yes", ["A"]), ("B yes", ["B"]),
    ("A yeah", ["A"]), ("B yeah", ["B"]),
    ("definitely A", ["A"]), ("definitely B", ["B"]),
    ("A definitely", ["A"]), ("B definitely", ["B"]),
    ("absolutely A", ["A"]), ("absolutely B", ["B"]),
    ("for sure A", ["A"]), ("for sure B", ["B"]),
    ("100% A", ["A"]), ("100% B", ["B"]),
    ("bingo A", ["A"]), ("A for sure", ["A"]), ("B no doubt", ["B"]),
    ("yes the left one", ["A"]), ("yes the right one", ["B"]),
    ("definitely the first", ["A"]), ("absolutely the second", ["B"]),
]
for utt, lab in CONF_WITH_REF_2OPT:
    NEW_EXAMPLES_V3.append({
        "utterance": utt,
        "options": ["A", "B"],
        "label": lab,
        "category": "confirmation_with_referent",
    })

CONF_WITH_REF_4OPT = [
    ("yes A", ["A"]), ("yeah C", ["C"]), ("definitely D", ["D"]),
    ("A yes", ["A"]), ("D yeah", ["D"]), ("C definitely", ["C"]),
    ("yes A and B", ["A", "B"]), ("yeah C and D", ["C", "D"]),
    ("definitely A and D", ["A", "D"]),
    ("absolutely B", ["B"]), ("100% D", ["D"]),
    ("bingo, D", ["D"]), ("spot on, its A", ["A"]),
    ("exactly, C", ["C"]), ("right on, B", ["B"]),
    ("yes the first one", ["A"]), ("yeah the last one", ["D"]),
    ("definitely the third", ["C"]),
    ("A yes for sure", ["A"]), ("C definitely for sure", ["C"]),
    ("yes A B and D", ["A", "B", "D"]),
    ("yeah all of them", ["A", "B", "C", "D"]),
    ("definitely both A and B", ["A", "B"]),
    ("absolutely, the middle two", ["B", "C"]),
    ("for sure, first and last", ["A", "D"]),
    ("yep A and C look right", ["A", "C"]),
    ("exactly, B is it", ["B"]),
    ("no doubt about A", ["A"]),
    ("100 percent D", ["D"]),
    ("agreed, C", ["C"]),
    ("sold on B", ["B"]),
    ("count me in for A", ["A"]),
    ("im in for D", ["D"]),
    ("A totally", ["A"]),
    ("B without a doubt", ["B"]),
]
for utt, lab in CONF_WITH_REF_4OPT:
    NEW_EXAMPLES_V3.append({
        "utterance": utt,
        "options": ["A", "B", "C", "D"],
        "label": lab,
        "category": "confirmation_with_referent",
    })

# --- 3) ambient_reinforce (more of the v2 pattern) ---
AMBIENT_REINFORCE = [
    "wow", "wow nice", "oh wow", "hah nice", "haha okay",
    "neat", "neat!", "cute", "aight", "aight cool",
    "chill", "chill vibes", "vibes", "big vibes",
    "sick", "dope", "based", "clean", "smooth", "slick",
    "top notch", "top tier", "solid stuff", "quality",
    "beautiful", "gorgeous", "stunning", "impressive",
    "yeah nice one", "oh sweet", "great stuff",
]
for utt in AMBIENT_REINFORCE:
    NEW_EXAMPLES_V3.append({
        "utterance": utt,
        "options": ["A", "B", "C", "D"],
        "label": "*",
        "category": "ambient_positive",
    })

print(f"total new examples: {len(NEW_EXAMPLES_V3)}")
print(f"by category: {Counter(ex['category'] for ex in NEW_EXAMPLES_V3)}")
def label_kind(l):
    if l == "*": return "star"
    if l == []: return "empty"
    if isinstance(l, list) and len(l) == 1: return "single"
    if isinstance(l, list) and len(l) == 2: return "two"
    return "three_plus"
print(f"by label kind: {Counter(label_kind(ex['label']) for ex in NEW_EXAMPLES_V3)}")

total new examples: 152
by category: Counter({'confirmation_with_referent': 61, 'confirmation_bare': 60, 'ambient_positive': 31})
by label kind: Counter({'star': 91, 'single': 52, 'two': 7, 'three_plus': 2})


In [5]:
import shutil
from datetime import datetime

# reload current train.jsonl (which already has v2's 173 additions)
train_current = load_jsonl(DATA / "train_v2.jsonl")
print(f"current train.jsonl: {len(train_current)} examples")

# drop any duplicates against existing utterances
existing_utts = {ex["utterance"].lower().strip() for ex in train_current}
NEW_CLEAN = [
    ex for ex in NEW_EXAMPLES_V3
    if ex["utterance"].lower().strip() not in existing_utts
]
print(f"after dedupe vs train.jsonl: {len(NEW_CLEAN)} (dropped {len(NEW_EXAMPLES_V3) - len(NEW_CLEAN)})")

# also dedupe within the new set (same utterance appearing twice with same opts+label)
seen = set()
NEW_FINAL = []
for ex in NEW_CLEAN:
    key = (ex["utterance"].lower().strip(), tuple(ex["options"]), json.dumps(ex["label"]))
    if key in seen:
        continue
    seen.add(key)
    NEW_FINAL.append(ex)
print(f"after internal dedupe: {len(NEW_FINAL)}")

# train_v3.jsonl = train_v2.jsonl + new examples
train_path = DATA / "train_v3.jsonl"
shutil.copy(DATA / "train_v2.jsonl", train_path)

# append
with train_path.open("a", encoding="utf-8") as f:
    for ex in NEW_FINAL:
        f.write(json.dumps(ex) + "\n")

# verify
train_v3 = load_jsonl(train_path)
print(f"\ntrain.jsonl now: {len(train_v3)} (was {len(train_current)}, delta +{len(train_v3) - len(train_current)})")
print(f"\nfinal category counts:")
print(Counter(ex.get("category", "unknown") for ex in train_v3))

current train.jsonl: 1451 examples
after dedupe vs train.jsonl: 147 (dropped 5)
after internal dedupe: 147
backed up to: train.jsonl.bak-20260903-002204

train.jsonl now: 1598 (was 1451, delta +147)

final category counts:
Counter({'negation': 368, 'reject_all': 175, 'mixed_sentiment': 171, 'comparative': 158, 'off_topic': 132, 'uncertainty': 132, 'multi_positive': 121, 'ambient_positive': 93, 'single_positive': 60, 'confirmation_bare': 59, 'confirmation_with_referent': 57, 'position_words': 42, 'ambient_negative': 30})


In [6]:
free_gpu()

from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig

MODEL_3B = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_3B_V3 = CHECKPOINTS / "qwen3b-qlora-v3_bad_do_not_use"

# reload train data fresh
train_data_full = load_jsonl(DATA / "train_v3.jsonl")
print(f"training on {len(train_data_full)} examples")

# format as chat messages using SYSTEM_PROMPT_V4
def to_chat(ex):
    options_str = "[" + ", ".join(f'"{o}"' for o in ex["options"]) + "]"
    user_msg = f"Options: {options_str}\nUser: {ex['utterance']}"
    assistant_msg = json.dumps(ex["label"])
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT_V4},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]}

train_ds = Dataset.from_list([to_chat(ex) for ex in train_data_full])
print(f"dataset size: {len(train_ds)}")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL_3B)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_3B, quantization_config=bnb, device_map="cuda",
)
base = prepare_model_for_kbit_training(base)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

sft_cfg = SFTConfig(
    output_dir=str(OUTPUT_3B_V3),
    num_train_epochs=4,                       # bumped from 3
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="steps",
    save_steps=50,                            # save often, hang-safe
    save_total_limit=4,
    bf16=False,                               # Blackwell fix
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=768,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    processing_class=tok,
    args=sft_cfg,
)

print("starting training...")
trainer.train()
print("done. saving final adapter...")
trainer.save_model(str(OUTPUT_3B_V3 / "checkpoint-final"))
print(f"saved to {OUTPUT_3B_V3}")

del model, base, trainer, tok
free_gpu()

VRAM: 0.00 GB
training on 1598 examples
dataset size: 1598


W0903 00:24:12.429000 22516 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


Tokenizing train dataset:   0%|          | 0/1598 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1598 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


starting training...


Step,Training Loss
5,3.392247
10,2.311574
15,1.680541
20,0.850965
25,0.146096
30,0.058161
35,0.049314
40,0.039341
45,0.039699
50,0.037095


done. saving final adapter...
saved to C:\Users\shlok\projects\ddp-llm\parser\checkpoints\qwen3b-qlora-v3
VRAM: 1.26 GB


In [7]:
free_gpu()

ADAPTER_3B_V3 = CHECKPOINTS / "qwen3b-qlora-v3_bad_do_not_use" / "checkpoint-final"
assert ADAPTER_3B_V3.exists(), f"missing: {ADAPTER_3B_V3}"

test_data = load_jsonl(DATA / "test.jsonl")
print(f"loaded {len(test_data)} test examples")

# Extended adversarial set: original 23 + the 17 words that failed on v2 in the
# app testing. If v3 handles this bigger set well, we're actually done.
ADVERSARIAL_V3 = [
    # -- original v2 adversarial (should all still work) --
    ("works great",    "*"),
    ("looks nice",     "*"),
    ("hmm",            "*"),
    ("okay",           "*"),
    ("meh",            "*"),
    ("not sure",       "*"),
    ("dunno really",   "*"),
    ("cool",           "*"),
    ("sure whatever",  "*"),
    ("interesting",    "*"),
    ("i guess",        "*"),
    ("perfect",        "*"),
    ("nice one",       "*"),
    ("go with it",     "*"),
    ("that works",     "*"),
    # -- confirmation words that failed after v2 (this retrain targets these) --
    ("definitely",     "*"),
    ("absolutely yes", "*"),
    ("for sure",       "*"),
    ("yes",            "*"),
    ("yeah",           "*"),
    ("yep",            "*"),
    ("exactly",        "*"),
    ("spot on",        "*"),
    ("right on",       "*"),
    ("bingo",          "*"),
    # -- sanity checks: same words WITH referent should still endorse --
    ("A",              ["A"]),
    ("B",              ["B"]),
    ("yes A",          ["A"]),
    ("definitely B",   ["B"]),
    ("A definitely",   ["A"]),
    ("the left one",   ["A"]),
    ("the right one",  ["B"]),
    ("neither",        []),
    ("none of them",   []),
    ("both are fine",  ["A", "B"]),
    ("A and B both work", ["A", "B"]),
]

print(f"\n{'='*60}\nloading 3B-v3...\n{'='*60}", flush=True)
model, tok = build_model("Qwen/Qwen2.5-3B-Instruct", ADAPTER_3B_V3)
print(f"VRAM after load (GB): {torch.cuda.memory_allocated() / 1e9:.2f}", flush=True)


# --- adversarial ---
print(f"\n=== 3B-v3: 2-option adversarial (extended) ===")
n_ambig = n_correct_ambig = n_sanity = n_correct_sanity = 0
for utt, gold in ADVERSARIAL_V3:
    parsed, raw = parse(model, tok, utt, ("A", "B"))
    ok = labels_equal(parsed, gold)
    is_ambig = (gold == "*")
    if is_ambig:
        n_ambig += 1
        if ok: n_correct_ambig += 1
    else:
        n_sanity += 1
        if ok: n_correct_sanity += 1
    mark = "OK  " if ok else "FAIL"
    tag = "AMB" if is_ambig else "SAN"
    print(f"  [{mark}] [{tag}] {utt!r:22s} gold={gold!r:14s} got={parsed!r}")
print(f"\nambiguous:  {n_correct_ambig}/{n_ambig} = {100*n_correct_ambig/n_ambig:.1f}%")
print(f"sanity:     {n_correct_sanity}/{n_sanity} = {100*n_correct_sanity/n_sanity:.1f}%")


# --- full test set (regression check) ---
print(f"\n=== 3B-v3: full test set ===")
correct = 0
per_cat = {}
failures = []
for i, ex in enumerate(test_data):
    if i % 40 == 0:
        print(f"  test progress: {i}/{len(test_data)}", flush=True)
    parsed, raw = parse(model, tok, ex["utterance"], ex["options"])
    ok = labels_equal(parsed, ex["label"])
    cat = ex.get("category", "unknown")
    per_cat.setdefault(cat, [0, 0])
    per_cat[cat][1] += 1
    if ok:
        correct += 1
        per_cat[cat][0] += 1
    else:
        failures.append((ex, parsed, raw))
print(f"\noverall: {correct}/{len(test_data)} = {100*correct/len(test_data):.1f}%")
for cat in sorted(per_cat):
    c, t = per_cat[cat]
    print(f"  {cat:20s} {c}/{t} = {100*c/t:.1f}%")
if failures:
    print(f"failures ({len(failures)}):")
    for ex, parsed, raw in failures[:15]:
        print(f"  utt={ex['utterance']!r}  gold={ex['label']!r}  got={parsed!r}")

del model, tok
free_gpu()

VRAM: 1.26 GB
loaded 210 test examples

loading 3B-v3...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


VRAM after load (GB): 3.45

=== 3B-v3: 2-option adversarial (extended) ===


C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  [FAIL] [AMB] 'works great'          gold='*'            got=['A', 'B']
  [OK  ] [AMB] 'looks nice'           gold='*'            got='*'
  [OK  ] [AMB] 'hmm'                  gold='*'            got='*'
  [FAIL] [AMB] 'okay'                 gold='*'            got=['A', 'B']
  [OK  ] [AMB] 'meh'                  gold='*'            got='*'
  [OK  ] [AMB] 'not sure'             gold='*'            got='*'
  [OK  ] [AMB] 'dunno really'         gold='*'            got='*'
  [OK  ] [AMB] 'cool'                 gold='*'            got='*'
  [FAIL] [AMB] 'sure whatever'        gold='*'            got=['A', 'B']
  [OK  ] [AMB] 'interesting'          gold='*'            got='*'
  [FAIL] [AMB] 'i guess'              gold='*'            got=['A', 'B']
  [FAIL] [AMB] 'perfect'              gold='*'            got=['A', 'B']
  [FAIL] [AMB] 'nice one'             gold='*'            got=['A', 'B']
  [FAIL] [AMB] 'go with it'           gold='*'            got=['A', 'B']
  [FAIL] [AMB] 'that works'

In [8]:
free_gpu()

# Load fresh
model, tok = build_model("Qwen/Qwen2.5-3B-Instruct", ADAPTER_3B_V3)

# The v2 prompt (from previous notebook — no confirmation-words clause)
SYSTEM_PROMPT_V3_APP = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.
- CRITICAL: If the utterance is only a short positive reaction that does NOT reference any option (e.g. "works great", "perfect", "nice one", "go with it", "that works", "cool"), output "*". A positive word alone is not enough. To count as an endorsement, the user must reference the options - either by letter (A, B, C, D), position (left, right, first, second, middle, last), quantifier over options (all, both, either, these, the ones, all four, both of them), or description that clearly points at an option.

Output ONLY the JSON. No explanation, no prose."""

print(f"\n=== 3B-v3 model + v3 prompt (no confirmation-words clause) ===")
n_ambig = n_correct_ambig = n_sanity = n_correct_sanity = 0
for utt, gold in ADVERSARIAL_V3:
    parsed, raw = parse(model, tok, utt, ("A", "B"), system_prompt=SYSTEM_PROMPT_V3_APP)
    ok = labels_equal(parsed, gold)
    is_ambig = (gold == "*")
    if is_ambig:
        n_ambig += 1
        if ok: n_correct_ambig += 1
    else:
        n_sanity += 1
        if ok: n_correct_sanity += 1
    mark = "OK  " if ok else "FAIL"
    tag = "AMB" if is_ambig else "SAN"
    print(f"  [{mark}] [{tag}] {utt!r:22s} gold={gold!r:14s} got={parsed!r}")
print(f"\nambiguous:  {n_correct_ambig}/{n_ambig} = {100*n_correct_ambig/n_ambig:.1f}%")
print(f"sanity:     {n_correct_sanity}/{n_sanity} = {100*n_correct_sanity/n_sanity:.1f}%")

del model, tok
free_gpu()

VRAM: 1.26 GB


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


=== 3B-v3 model + v3 prompt (no confirmation-words clause) ===
  [OK  ] [AMB] 'works great'          gold='*'            got='*'
  [OK  ] [AMB] 'looks nice'           gold='*'            got='*'
  [OK  ] [AMB] 'hmm'                  gold='*'            got='*'
  [OK  ] [AMB] 'okay'                 gold='*'            got='*'
  [OK  ] [AMB] 'meh'                  gold='*'            got='*'
  [OK  ] [AMB] 'not sure'             gold='*'            got='*'
  [OK  ] [AMB] 'dunno really'         gold='*'            got='*'
  [OK  ] [AMB] 'cool'                 gold='*'            got='*'
  [FAIL] [AMB] 'sure whatever'        gold='*'            got=['A', 'B']
  [OK  ] [AMB] 'interesting'          gold='*'            got='*'
  [OK  ] [AMB] 'i guess'              gold='*'            got='*'
  [OK  ] [AMB] 'perfect'              gold='*'            got='*'
  [FAIL] [AMB] 'nice one'             gold='*'            got=['A', 'B']
  [FAIL] [AMB] 'go with it'           gold='*'            got=['

In [9]:
free_gpu()

ADAPTER_3B_V2 = CHECKPOINTS / "qwen3b-qlora-v2" / "checkpoint-final"
assert ADAPTER_3B_V2.exists()

# The v3 app prompt (v2 model's known-good prompt)
SYSTEM_PROMPT_V3_APP = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.
- CRITICAL: If the utterance is only a short positive reaction that does NOT reference any option (e.g. "works great", "perfect", "nice one", "go with it", "that works", "cool"), output "*". A positive word alone is not enough. To count as an endorsement, the user must reference the options - either by letter (A, B, C, D), position (left, right, first, second, middle, last), quantifier over options (all, both, either, these, the ones, all four, both of them), or description that clearly points at an option.

Output ONLY the JSON. No explanation, no prose."""

# The BIG adversarial set: everything we've tested plus more variations.
# Grouped so we can see failure clusters.
ADVERSARIAL_FULL = [
    # -- group 1: quality/reaction words (v2's strength) --
    ("works great",    "*"), ("looks nice",     "*"), ("looks good",     "*"),
    ("looks fine",     "*"), ("looks great",    "*"), ("seems nice",     "*"),
    ("cool",           "*"), ("nice one",       "*"), ("perfect",        "*"),
    ("great",          "*"), ("awesome",        "*"), ("amazing",        "*"),
    ("brilliant",      "*"), ("beautiful",      "*"), ("love it",        "*"),
    ("go with it",     "*"), ("that works",     "*"), ("that'll do",     "*"),
    ("solid",          "*"), ("neat",           "*"),

    # -- group 2: bare confirmations (v2's weakness) --
    ("yes",            "*"), ("yeah",           "*"), ("yep",            "*"),
    ("yup",            "*"), ("definitely",     "*"), ("absolutely",     "*"),
    ("absolutely yes", "*"), ("for sure",       "*"), ("sure",           "*"),
    ("of course",      "*"), ("100%",           "*"), ("totally",        "*"),
    ("exactly",        "*"), ("spot on",        "*"), ("right on",       "*"),
    ("bingo",          "*"), ("no doubt",       "*"), ("agreed",         "*"),

    # -- group 3: uncertainty / fillers (should be easy) --
    ("hmm",            "*"), ("okay",           "*"), ("meh",            "*"),
    ("not sure",       "*"), ("dunno really",   "*"), ("sure whatever",  "*"),
    ("interesting",    "*"), ("i guess",        "*"), ("alright",        "*"),

    # -- group 4: negative fillers (should be *, not []) --
    ("eh",             "*"), ("not great",      "*"), ("boring",         "*"),
    ("bland",          "*"), ("whatever",       "*"), ("nothing special","*"),

    # -- SANITY: with referent, should still endorse --
    ("A",              ["A"]), ("B",              ["B"]),
    ("yes A",          ["A"]), ("definitely B",   ["B"]),
    ("A definitely",   ["A"]), ("yeah B",         ["B"]),
    ("the left one",   ["A"]), ("the right one",  ["B"]),
    ("first please",   ["A"]), ("second please",  ["B"]),
    ("both are fine",  ["A", "B"]), ("either",     ["A", "B"]),
    ("A and B",        ["A", "B"]),
    ("neither",        []), ("none of them",   []),
    ("nope",           []), ("both wrong",     []),
]

print(f"loading 3B-v2...")
model, tok = build_model("Qwen/Qwen2.5-3B-Instruct", ADAPTER_3B_V2)
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print(f"\n=== 3B-v2 + v3 prompt: full audit ({len(ADVERSARIAL_FULL)} cases) ===\n")

# collect by group
groups = {
    "1. quality/reaction": ADVERSARIAL_FULL[0:20],
    "2. bare confirmations": ADVERSARIAL_FULL[20:38],
    "3. uncertainty/fillers": ADVERSARIAL_FULL[38:47],
    "4. negative fillers": ADVERSARIAL_FULL[47:53],
    "5. sanity (referent)": ADVERSARIAL_FULL[53:],
}

all_failures = []
for group_name, cases in groups.items():
    print(f"--- {group_name} ---")
    n_ok = 0
    for utt, gold in cases:
        parsed, raw = parse(model, tok, utt, ("A", "B"), system_prompt=SYSTEM_PROMPT_V3_APP)
        ok = labels_equal(parsed, gold)
        if ok:
            n_ok += 1
        else:
            all_failures.append((group_name, utt, gold, parsed))
        mark = "OK  " if ok else "FAIL"
        print(f"  [{mark}] {utt!r:20s} gold={gold!r:14s} got={parsed!r}")
    print(f"  -> {n_ok}/{len(cases)}\n")

print(f"\n=== FAILURE SUMMARY ===")
print(f"total failures: {len(all_failures)} / {len(ADVERSARIAL_FULL)}")
if all_failures:
    print(f"\nby group:")
    from collections import defaultdict
    by_group = defaultdict(list)
    for g, u, gold, got in all_failures:
        by_group[g].append((u, gold, got))
    for g, items in by_group.items():
        print(f"\n  {g} ({len(items)} failures):")
        for u, gold, got in items:
            print(f"    {u!r:20s} gold={gold!r:14s} got={got!r}")

del model, tok
free_gpu()

VRAM: 1.26 GB
loading 3B-v2...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

VRAM: 3.45 GB

=== 3B-v2 + v3 prompt: full audit (70 cases) ===

--- 1. quality/reaction ---
  [OK  ] 'works great'        gold='*'            got='*'
  [OK  ] 'looks nice'         gold='*'            got='*'
  [OK  ] 'looks good'         gold='*'            got='*'
  [OK  ] 'looks fine'         gold='*'            got='*'
  [OK  ] 'looks great'        gold='*'            got='*'
  [OK  ] 'seems nice'         gold='*'            got='*'
  [OK  ] 'cool'               gold='*'            got='*'
  [OK  ] 'nice one'           gold='*'            got='*'
  [OK  ] 'perfect'            gold='*'            got='*'
  [OK  ] 'great'              gold='*'            got='*'
  [OK  ] 'awesome'            gold='*'            got='*'
  [OK  ] 'amazing'            gold='*'            got='*'
  [OK  ] 'brilliant'          gold='*'            got='*'
  [OK  ] 'beautiful'          gold='*'            got='*'
  [OK  ] 'love it'            gold='*'            got='*'
  [OK  ] 'go with it'         gold='*

In [10]:
free_gpu()

ADAPTER_3B_V2 = CHECKPOINTS / "qwen3b-qlora-v2" / "checkpoint-final"
assert ADAPTER_3B_V2.exists()

SYSTEM_PROMPT_V3_APP = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.
- CRITICAL: If the utterance is only a short positive reaction that does NOT reference any option (e.g. "works great", "perfect", "nice one", "go with it", "that works", "cool"), output "*". A positive word alone is not enough. To count as an endorsement, the user must reference the options - either by letter (A, B, C, D), position (left, right, first, second, middle, last), quantifier over options (all, both, either, these, the ones, all four, both of them), or description that clearly points at an option.

Output ONLY the JSON. No explanation, no prose."""

# Ordinal test cases - across N=2, 3, 4 options.
# The verbalizer's "ordinal" style emits things like "the first one" and
# "the second one" for a 2-item query. Broader coverage below.
ORDINAL_CASES = [
    # -- N=2 (what the verbalizer actually produces) --
    ("the first one",       ("A", "B"),           ["A"]),
    ("the second one",      ("A", "B"),           ["B"]),
    ("first",               ("A", "B"),           ["A"]),
    ("second",              ("A", "B"),           ["B"]),
    ("first one",           ("A", "B"),           ["A"]),
    ("second one",          ("A", "B"),           ["B"]),
    ("the 1st",             ("A", "B"),           ["A"]),
    ("the 2nd",             ("A", "B"),           ["B"]),
    ("#1",                  ("A", "B"),           ["A"]),
    ("#2",                  ("A", "B"),           ["B"]),
    ("number one",          ("A", "B"),           ["A"]),
    ("number two",          ("A", "B"),           ["B"]),

    # -- N=3 --
    ("the first",           ("A", "B", "C"),      ["A"]),
    ("the second",          ("A", "B", "C"),      ["B"]),
    ("the third",           ("A", "B", "C"),      ["C"]),
    ("first one",           ("A", "B", "C"),      ["A"]),
    ("second one",          ("A", "B", "C"),      ["B"]),
    ("third one",           ("A", "B", "C"),      ["C"]),
    ("the last one",        ("A", "B", "C"),      ["C"]),
    ("the middle one",      ("A", "B", "C"),      ["B"]),

    # -- N=4 --
    ("the first",           ("A", "B", "C", "D"), ["A"]),
    ("the second",          ("A", "B", "C", "D"), ["B"]),
    ("the third",           ("A", "B", "C", "D"), ["C"]),
    ("the fourth",          ("A", "B", "C", "D"), ["D"]),
    ("first one please",    ("A", "B", "C", "D"), ["A"]),
    ("fourth one please",   ("A", "B", "C", "D"), ["D"]),
    ("the last one",        ("A", "B", "C", "D"), ["D"]),
    ("the last",            ("A", "B", "C", "D"), ["D"]),

    # -- ranges / multi-ordinals --
    ("the first and third", ("A", "B", "C", "D"), ["A", "C"]),
    ("first and last",      ("A", "B", "C", "D"), ["A", "D"]),
    ("first two",           ("A", "B", "C", "D"), ["A", "B"]),
    ("last two",            ("A", "B", "C", "D"), ["C", "D"]),
    ("second and fourth",   ("A", "B", "C", "D"), ["B", "D"]),
    ("first three",         ("A", "B", "C", "D"), ["A", "B", "C"]),
    ("last three",          ("A", "B", "C", "D"), ["B", "C", "D"]),

    # -- combined with confirmation / reaction (should still work) --
    ("yes the first",       ("A", "B"),           ["A"]),
    ("go with the second",  ("A", "B"),           ["B"]),
    ("the first works",     ("A", "B"),           ["A"]),
    ("the second please",   ("A", "B"),           ["B"]),
]

print(f"loading 3B-v2...")
model, tok = build_model("Qwen/Qwen2.5-3B-Instruct", ADAPTER_3B_V2)
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB\n")

print(f"=== 3B-v2 + v3 prompt: ordinal audit ({len(ORDINAL_CASES)} cases) ===\n")

by_n = {2: [], 3: [], 4: []}
for utt, opts, gold in ORDINAL_CASES:
    by_n[len(opts)].append((utt, opts, gold))

all_failures = []
for n in [2, 3, 4]:
    print(f"--- N={n} options ---")
    n_ok = 0
    for utt, opts, gold in by_n[n]:
        parsed, raw = parse(model, tok, utt, opts, system_prompt=SYSTEM_PROMPT_V3_APP)
        ok = labels_equal(parsed, gold)
        if ok:
            n_ok += 1
        else:
            all_failures.append((n, utt, opts, gold, parsed))
        mark = "OK  " if ok else "FAIL"
        print(f"  [{mark}] N={n} {utt!r:28s} gold={gold!r:18s} got={parsed!r}")
    print(f"  -> {n_ok}/{len(by_n[n])}\n")

print(f"\n=== FAILURE SUMMARY ===")
print(f"total failures: {len(all_failures)} / {len(ORDINAL_CASES)}")
if all_failures:
    print(f"\ndetails:")
    for n, u, opts, gold, got in all_failures:
        print(f"  N={n} {u!r:28s} gold={gold!r:18s} got={got!r}")

del model, tok
free_gpu()

VRAM: 1.26 GB
loading 3B-v2...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

VRAM: 3.45 GB

=== 3B-v2 + v3 prompt: ordinal audit (39 cases) ===

--- N=2 options ---
  [OK  ] N=2 'the first one'              gold=['A']              got=['A']
  [OK  ] N=2 'the second one'             gold=['B']              got=['B']
  [OK  ] N=2 'first'                      gold=['A']              got=['A']
  [OK  ] N=2 'second'                     gold=['B']              got=['B']
  [OK  ] N=2 'first one'                  gold=['A']              got=['A']
  [OK  ] N=2 'second one'                 gold=['B']              got=['B']
  [OK  ] N=2 'the 1st'                    gold=['A']              got=['A']
  [OK  ] N=2 'the 2nd'                    gold=['B']              got=['B']
  [OK  ] N=2 '#1'                         gold=['A']              got=['A']
  [OK  ] N=2 '#2'                         gold=['B']              got=['B']
  [OK  ] N=2 'number one'                 gold=['A']              got=['A']
  [OK  ] N=2 'number two'                 gold=['B']              got=['B']
